# 02 — Clean

Turn the raw scraped tables into one clean, analysis-ready film table inside DuckDB, then
save it to `data/interim/` as Parquet.

**Steps (all in DuckDB):**
1. Parse The Numbers' dollar strings (`$2,717,503,922` → integer) and release date to a DATE.
2. Filter to the **released theatrical** universe — worldwide gross > 0 (drops the announced
   $0 films and the *Werewolf by Night* TV special).
3. Join the `phase_map` lookup (film → phase → saga). The join is exact (verified in `01`).
4. Cross-check domestic gross against Box Office Mojo's independent franchise figures.
5. Quality report, then save `mcu_films_clean` to interim Parquet.

Grosses are **nominal** (year-of-release dollars) — this project's story is *share of the
franchise*, not cross-era ranking, so no inflation adjustment is applied (documented in
`SOURCES.md`).

In [ ]:
import sys, os
from pathlib import Path

PROJECT = Path.cwd()
while not (PROJECT / 'config.yaml').exists() and PROJECT != PROJECT.parent:
    PROJECT = PROJECT.parent
os.chdir(PROJECT)
sys.path.insert(0, str(PROJECT))

import pandas as pd
from src.ingest import load_config
from src.clean_quality import get_connection, run_sql, quality_report, save_interim

cfg = load_config('config.yaml')
con = get_connection(cfg)
print('raw tables:', run_sql("SELECT table_name FROM information_schema.tables "
                             "WHERE table_name LIKE '%_raw' OR table_name='phase_map'", con)['table_name'].tolist())

## Parse, filter, and join → `mcu_films_clean`

One SQL statement does the heavy lifting: strip `$` and commas and cast the four money columns
to `BIGINT`, parse the release date, keep only rows with a positive worldwide gross (the
released theatrical films), and inner-join the Phase/Saga map. `production_budget` is kept
where present (a few films report `$0` budget on The Numbers — treated as NULL/unknown, not a
real zero).

In [ ]:
SQL = '''
WITH parsed AS (
    SELECT
        title,
        TRY_STRPTIME(release_date, '%b %d, %Y')::DATE            AS release_date,
        TRY_CAST(REPLACE(REPLACE(worldwide_box_office, '$',''), ',','') AS BIGINT) AS worldwide_gross,
        TRY_CAST(REPLACE(REPLACE(domestic_box_office,  '$',''), ',','') AS BIGINT) AS domestic_gross,
        TRY_CAST(REPLACE(REPLACE(opening_weekend,      '$',''), ',','') AS BIGINT) AS opening_weekend,
        NULLIF(TRY_CAST(REPLACE(REPLACE(production_budget,'$',''), ',','') AS BIGINT), 0) AS production_budget
    FROM tnumbers_mcu_raw
)
SELECT
    p.title,
    m.phase_num,
    m.phase,
    m.saga,
    p.release_date,
    EXTRACT(year FROM p.release_date)::INTEGER AS release_year,
    p.worldwide_gross,
    p.domestic_gross,
    (p.worldwide_gross - p.domestic_gross)     AS international_gross,
    ROUND(p.domestic_gross::DOUBLE / p.worldwide_gross, 4)     AS domestic_share,
    p.opening_weekend,
    p.production_budget
FROM parsed p
JOIN phase_map m ON m.film = p.title
WHERE p.worldwide_gross > 0
ORDER BY p.release_date
'''
df = run_sql(SQL, con)
print('clean film rows:', len(df))
df[['title','phase','saga','release_year','worldwide_gross','domestic_gross','domestic_share']].head(40)

Confirm the filter dropped exactly the non-theatrical / unreleased rows and that every
released film got a phase (no NULLs from the join).

In [ ]:
raw_n = run_sql('SELECT COUNT(*) n FROM tnumbers_mcu_raw', con)['n'][0]
print(f'raw rows: {raw_n}  ->  clean rows: {len(df)}  (dropped {raw_n - len(df)})')
print('films per saga / phase:')
display(df.groupby(['saga','phase'], sort=False).agg(films=('title','size'),
        worldwide=('worldwide_gross','sum')))
assert df['phase'].notna().all(), 'some films missing a phase!'
assert df['worldwide_gross'].gt(0).all(), 'zero-gross row slipped through!'
assert len(df) == df['title'].nunique(), 'duplicate titles!'
print('\nintegrity checks passed')

## Cross-check domestic gross against Box Office Mojo

The treemap sizes tiles by **worldwide** gross (only The Numbers has that per film), but we can
still validate The Numbers against an independent source on the **domestic** figure. Box Office
Mojo's franchise page lists domestic lifetime gross per film; we parse it, drop its re-release
and "Columbia 100th Anniversary" rows, and compare the two sources film by film. Small
differences (reporting cutoffs, re-release accounting) are expected; large ones would flag a
data problem.

In [ ]:
BOM_SQL = '''
SELECT
    release AS title,
    TRY_CAST(REPLACE(REPLACE(lifetime_gross, '$',''), ',','') AS BIGINT) AS bom_domestic
FROM bom_mcu_raw
WHERE release NOT ILIKE '%Re-release%'
  AND release NOT ILIKE '%Columbia 100th%'
'''
bom = run_sql(BOM_SQL, con)
chk = df[['title','domestic_gross']].merge(bom, on='title', how='inner')
chk['pct_diff'] = ((chk['domestic_gross'] - chk['bom_domestic']) / chk['bom_domestic'] * 100).round(2)
print(f'matched {len(chk)} of {len(df)} films to BOM')
print(f'max abs % diff on domestic gross: {chk["pct_diff"].abs().max():.2f}%')
chk.reindex(chk['pct_diff'].abs().sort_values(ascending=False).index).head(10)

The two independent sources agree on domestic gross to within a fraction of a percent for the
overwhelming majority of films (any larger gaps are re-release accounting, noted in `SOURCES.md`).
The Numbers is retained as the single source of truth so domestic and worldwide come from one
consistent place.

## Quality report + save interim

Load the clean table back into DuckDB as `mcu_films_clean` and save it to
`data/interim/mcu_films_clean.parquet`.

In [ ]:
from src.clean_quality import load_to_duckdb
load_to_duckdb(df, 'mcu_films_clean', con)
_ = quality_report(df, 'mcu_films_clean', con,
                   required_columns=['title','phase','saga','worldwide_gross','domestic_gross'])

In [ ]:
save_interim(df, cfg, 'mcu_films_clean.parquet')

---
**Next:** `03-prepare.ipynb` — add share-of-franchise / share-of-phase columns and package
the export (CSV + Excel + Parquet + codebook).

---
## Cleanup
Close the DuckDB connection so the single-writer lock is released.

In [ ]:
con.close()
print('connection closed')